# Transport of a Pollutant in a River

## Introduction

A quantity of a pollutant has entered a river. At time $t=0$, its concentration profile is block-shaped, with a length of 10 km and a maximum concentration of 10 mg/$m^{3}$. For the river section, we have the following data: the average flow velocity is $u = 0.5$ m/s, and the diffusion coefficient is $K = 100$ $m^{2}/s$.

The objective of this assignment is to determine when the maximum concentration passes a point located 60 km downstream from the point where the pollutant was present at $t=0$. We shall refer to these two locations as B and A, respectively. Furthermore, we aim to determine the value of the maximum concentration at point B.

You are provided with the Python script containing the code for the solution of the following 1D advection-diffusion equation:

$$
  \frac{\partial c}{\partial t} + u \frac{\partial c}{\partial x} - K\frac{\partial^2 c}{\partial x^2} = 0\, , \quad x \in [0,L]
$$

where $c(x,t)$ is the concentration (in mg/$m^{3}$) and $L$ is the length of the river section (in meters). This equation describes the transport of the cloud of pollutant along the river at a constant speed while spreading in the river as time proceeds.

For the simulations and the analysis that we will carry out, the following step sizes will be used as baseline: $\Delta x = 0.4$ km and $\Delta t = 10$ minutes.

## Part I: find the solution

>**Task 1.1**
>
>Before we proceed with simulating the transport of the cloud of pollutant, we will first answer a few questions to get an idea of the problem.
>
>1. How long does it take for the peak of the cloud to pass point B?
>2. What will the profile of the cloud look like at point B and what is its maximum length? You may provide a schematic drawing.
>3. What will the maximum concentration in B be?
>
>To answer the questions above, you may use the exact solution for the **diffusion** equation ($u=0$) with a block-shaped initial cloud that is centered around $x=0$. This solution is given as:
>
>$$
>  c(x,t) = \frac{c_0}{2} \left [ \text{erf} \left ( \frac{0.5L_c-x}{2\sqrt{Kt}} \right ) + \text{erf} \left ( \frac{0.5L_c+x}{2\sqrt{Kt}} \right ) \right ]
>$$
>
>with $c_0$ the initial concentration and $L_c$ the initial cloud length.
>
> **Hint:** Use the exact solution for diffusion considering that the advection term does not change the level of the maximum concentration for given $t$. Think about at which $x$ the diffusion solution will have its maximum. You can evaluate the error function $\text{erf}(z)$ with `scipy.special.erf`.


In [ ]:
import scipy
import numpy as np
import matplotlib.pyplot as plt

c0 = 10
L = 60000
u = 0.5
Lc = 10000
K = 100

tend = L / u
x = 0


c = c0/2 * ( scipy.special.erf((0.5*Lc-x)/(2*np.sqrt(K*tend))) + scipy.special.erf((0.5*Lc+x)/(2*np.sqrt(K*tend))) )
print(c, tend)

### Python module for numerical solution to advection-diffusion equation

Here we outline the usage of the Python module `advdiff.py` in this folder, which contains the solver for the above presented advection-diffusion equation.

An initial condition $c(x,0) = c0(x)$ must be provided by the user such that $c0(0) = c0(L) = 0$, so that no boundary conditions need to be imposed.

**An important note here: choose $L$ such that the entire profile of the cloud transported along the river is completely visible in the plot.**

The use of function `advdiff` is as follows:

```
c = advdiff ( u, K, c0, L, dt, T, dxmethod, dtmethod, theta )
```

and the input parameters are defined as follows:
- `u` = flow velocity (in m/s)
- `K` = diffusion coefficient (in m$^{2}$/s)
- `c0` = initial condition (row vector)
- `L` = length of computational domain (in m)
- `dt` = time step (in s)
- `T` = simulation end time (in s)
- `dxmethod` = spatial discretization scheme
- `dtmethod` = time integration scheme
- `theta` = implicitness parameter ($0 \leq $ `theta` $\leq 1$)

The output parameter is `c` which contains the numerical solution of the advection-diffusion equation.
This variable is a 2D array `c[0:N,0:M]` with `N` the number of time steps and `M` the number of grid cells.
The grid is equidistant so that the grid points are uniformly distributed with a constant grid size $\Delta x = L/$`M`.

The row vector `c0` represents the initial value of $c(x,0)$ at uniformly spaced points in $[0,L]$.
Hence, the dimension of this vector is `c0[0:M]`.
Note that `c0` is a grid function and thus indirectly defines the computational grid. (The script therefore calculates the value of $\Delta x$ itself.)

The allowed values of `dxmethod` in this assignment are:
- `central` = second order central differences
- `upwind` = first order upwind scheme

These two methods only concern the advection term while the diffusion term is approximated exclusively with central differences.

The relevant value of `dtmethod` in this assignment is `'theta'` and represents the $\theta$-method, to be discussed in the following part.

The last argument of `advdiff` specifies the actual value of the implicitness parameter $\theta$. Special cases are
- `theta = 0.0` is forward Euler
- `theta = 0.5` is Crank-Nicolson
- `theta = 1.0` is backward Euler

**The code in the `.py` file is complete and does not need to be modified.**

Import `advdiff` in the cell below.

In [ ]:
from advdiff import advdiff

>**Task 1.2**
>
>In the next cell, we specify the initial condition `c0` of the cloud of pollutant which has a *block-shaped* profile with a length of 10 km.
>Note that the maximum concentration value is 10 $mg/m^{3}$.
>For the computational domain, we must choose a length that visualizes the propagation of the bell-curve, including its head and tail, through the domain all the way. Make an estimate of this length.
>
>Fill in the missing domain length and run the cell below to plot the initial condition.
>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# domain length
lenD = 100000

lenC = 10000.             # length of c0
dx = 400.                 # grid size

M = int(lenD/dx) + 1      # number of grid points
x = np.linspace(0,lenD,M) # array with coordinates of grid points

# initial condition
c0 = np.zeros(M)
c0[int(0.5 * lenC/dx):int(0.5 * lenC/dx) + int(lenC/dx) + 1] = 10.
plt.plot(x,c0,'k',label='initial')
plt.legend(loc='upper right')
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')

>**Task 1.3**
>
>We will now run the simulation using the script above. Enter the required data in the fields below and run the cell.


In [ ]:
# parameters for advdiff

u= 0.5
K= 100
dt= 600
Tend= 120000


# compute solution to transport equation
c=advdiff(u,K,c0,lenD,dt,Tend,'central','theta',0.5)

The cell below creates a plot of the solution at final time step. Here, `N` represents the number of time steps. Enter a value for `N` and generate the plot.

In [ ]:
# number of timesteps
N= 200

plt.plot(x,c[N,:],'r',label='numerical solution')
plt.legend(loc='upper left')
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')
plt.grid()
print(c[N].max())

Compare the numerical solution with your answers from **Task 1.1**.

>**Task 1.4**
>
>What are the Courant and mesh P&eacute;clet numbers?


## Part II: accuracy

### Time integration using the $\theta$-method

During the course you were introduced to the following two time integration methods, namely
- the forward Euler scheme and
- the backward Euler scheme.

Each of these methods has its advantages and disadvantages. Basically, the forward Euler is conditionally stable
while the backward Euler is unconditionally stable. On the other hand, the former is simple to implement and also
cheap in terms of function evaluations while the latter usually results into a system of equations that need
to be solved.
Both methods, however, are first order accurate in time, which could potentially be a disadvantage in many applications.
Now let us take a look at how we can improve this.

Consider the following first order ODE

$$
  \frac{dy}{dt} = f(y)
$$

where we seek for a solution $y(t)$ in the range $0 \leq t \leq T_{\text{end}}$.
This time frame is discretized into a number of steps with a step size of $\Delta t$:
$\{t_n = n\Delta t \,\,|\,\, n = 0, 1, 2, \cdots\}$.
Then, the **forward (explicit) Euler scheme** is given by

$$
  \frac{y^{n+1}-y^n}{\Delta t} = f(y^n)
$$

while the **backward (implicit) Euler scheme** reads

$$
  \frac{y^{n+1}-y^n}{\Delta t} = f(y^{n+1})
$$

Again, both schemes have a truncation error of $\tau_{\Delta t} = \mathcal{O}(\Delta t)$.

If we take the average of the forward and backward Euler schemes, we get

$$
  \frac{y^{n+1}-y^n}{\Delta t} = \frac 12 f(y^n) + \frac 12 f(y^{n+1})
$$

which is known as the **Crank-Nicolson scheme** or the **trapezoidal rule**.
Since the last term of the right-hand side needs to be evaluated based on the solution at the next time step $y^{n+1}$,
the Crank-Nicolson scheme is regarded as *implicit*. Like the backward Euler, this scheme has the disadvantage of the required
system of equations to be solved, but has the advantageous property of unconditional stability.

The reason behind averaging the forward and backward Euler schemes is to improve the accuracy of the resulting scheme.
Indeed, the truncation error of the Crank-Nicolson scheme is given as $\tau_{\Delta t} = \mathcal{O}(\Delta t^2)$.

All the three mentioned schemes above can be generalized to an arbitrary weighted average of $f(y^n)$ and $f(y^{n+1})$, as follows

$$
  \frac{y^{n+1}-y^n}{\Delta t} = \theta\, f(y^{n+1}) + \left ( 1 - \theta \right ) \,f(y^n) 
$$

with $0 \leq \theta \leq 1$ a weight parameter to be chosen. This scheme is called the $\theta$-method
and is explicit if $\theta = 0$ and implicit if $0 < \theta \leq 1$. The parameter $\theta$ is called the
**implicitness parameter** (last argument of the above script). In practice, we will be almost exclusively interested in either
- forward Euler ($\theta = 0$) or
- backward Euler ($\theta = 1$) or
- Crank-Nicolson ($\theta = 1/2$)

In the remainder of this assignment we will work with these three schemes.

>**Task 2.1**
>
>
>
>The goal of this assignment is to determine whether the explicit and implict Euler methods are indeed less accurate than the trapezoidal rule. To this end, first:
> - repeat the previous calculation using a time step of 10 minutes, followed by a calculation with a time step of 5 minutes. 
> 
> Keep central differences for the spatial discretization. Compare both plots and write down your conclusions based on this comparison.
>
>Next: 
> - Perform the calculation again using the last time step, but this time apply the explicit Euler method instead of the trapezoidal rule. 
> - Continue halving the time step until there is no longer any discernible difference between the results of the last two calculations.
> - Repeat the same procedure but for the implicit Euler method. Include your graph and the observed time.
>
>Compare your result with that of the trapezoidal rule and explicit Euler.
>
>Explain your findings based on the notion of *order of accuracy*.


In [ ]:
# Parameters for Task 2.1

# We do the first one for you!
dxmethod = 'central'
dtmethod = 'theta'

theta = 0.5

# replot the current result
plt.plot(x,c[N,:],'r',label='trapezoidal - dt = 10 min')
# reduce time step by factor 2
dt = dt / 2


# Now it's your turn! 
c=advdiff(u,K,c0,lenD,dt,Tend,'central','theta',0.5)
N= int(Tend/dt)
plt.plot(x,c[N,:],'b',label='trapezoidal - dt = 5 min')


# finalize the plot. Hint: plot them together on the same figure
plt.axis(np.array([0,lenD,-1,10]))
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')
plt.legend(loc = "upper left")
plt.grid()


In [ ]:
# Parameters for Task 2.1

# We do the first one for you!
dxmethod = 'central'
dtmethod = 'theta'

theta = 0.5

dt = 300
# replot the current result
N= int(Tend/dt)
c=advdiff(u,K,c0,lenD,dt,Tend,'central','theta',0)
plt.plot(x,c[N,:],'r',label='trapezoidal - dt = 10 min')

# reduce time step by factor 2
dt = dt / 2

# Now it's your turn! 
c=advdiff(u,K,c0,lenD,dt,Tend,'central','theta',0)
N= int(Tend/dt)
plt.plot(x,c[N,:],'b',label='trapezoidal - dt = 5 min')


# finalize the plot. Hint: plot them together on the same figure
plt.axis(np.array([0,lenD,-1,10]))
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')
plt.legend(loc = "upper left")
plt.grid()

## Part III: effect of numerical diffusion

In this part of the assignment we will investigate the impact of numerical diffusion.
In general, there are two contributions to this numerical diffusion:
- the first order upwind scheme
- the $\theta$-method

To ensure that only numerical diffusion is at play, we set $K = 0$, so that there is no physical diffusion.
The other parameters of the previous runs in part I remain unchanged.

Since $K=0$, we solves the advection equation

$$
  \frac{\partial c}{\partial t} + v\,\frac{\partial c}{\partial x} = 0
$$

which represents the propagation of the substance in the river while the block shape remains unchanged throughout the journey.
In this way we can use this exact propagated solution for comparison with the obtained numerical solutions.

>**Task 3.1**
>
>Consider the same initial block-shaped profile of the cloud, with a maximum concentration of 10 mg/$m^{3}$ and a length of 10 km. Interpret the code below used to plot the cloud as it passes point B, 60 km downstream.
>


In [ ]:
# true solution propagated

distance = 60000.
c1 = np.zeros(M)
c1[int(0.5 * lenC/dx + distance/dx):int(0.5 * lenC/dx + distance/dx) + int(lenC/dx) + 1] = 10.

plt.plot(x,c0,'k',label='initial')
plt.plot(x,c1,'g',label='exact')

# finalize the plot
plt.axis(np.array([0,100000,0,11]))
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')
plt.legend(loc = "upper right")
plt.grid()

>**Task 3.2**
>
>Use function `advdiff` to do the following three runs:
>- first order upwind and $\theta = 0$ (FTBS)
>- central differences and $\theta = 0.5$ (central + trapezoidal)
>- first order upwind and $\theta = 1.0$ (BTBS)
>
>For these runs, set $\Delta x = 400$ m, $\Delta t = 600$ s and $K = 0$ $m^{2}$/s.
>
>Include a graph containing the results of the three runs together with the analytical solution of **Task 3.1**. Clearly mark which curve belongs to which run.
>
>Answer the following questions in your report.
>1. Which of the schemes has the most numerical diffusion? Estimate the amount of this numerical diffusion.
>2. Which scheme has no numerical diffusion and why?
>3. Which of the two contributions to numerical diffusion has the largest impact?
>   - the first order upwind scheme or
>   - the implicit Euler method
>4. Which of the schemes gives a non-negative solution? And do you think this has any connection to the answer to question 2? Explain your reasoning.
>


In [ ]:
# parameters for Task 3.2
# we do the first one for you!

dtmethod = 'theta'
K = 0.

# plot true solution
plt.plot(x,c1,'g',label='exact')


# Now it's your turn!
### YOUR CODE HERE ###

# plot the result
plt.axis(np.array([0,lenD,-3,13]))
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')
plt.legend(loc = "upper left")
plt.grid()

>**Task 3.3**
>
>Carry out two runs with the FTCS scheme:
>- without diffusion
>- with diffusion $K = 75$ $m^{2}$/s
>
>In both cases, set $u = 0.5$ m/s, $\Delta x = 400$ m and $\sigma = 0.75$.
>
>Make a graph showing the results without diffusion and with diffusion.
>Also make sure that the solution at *final time step* does not leave the domain. Clearly mark the curves.
>
>What is the mesh P&eacute;clet number?
>
>Explain your findings.


In [ ]:
# parameters for Task 3.3

u = 0.5
K = 75.
dx = 400.
sigma = 0.75
distance=60000.

# compute the end time and time step
Tend = np.ceil(distance/u)
dt = np.floor(sigma * dx / u)
N = int(Tend/dt)

# solve the advection-diffusion equation using the FTCS scheme
# without diffusion

c = ### YOUR CODE HERE ###
plt.plot(x,c[N,:],'r',label='without diffusion')

# with diffusion

c = ### YOUR CODE HERE ###
plt.plot(x,c[N,:],'b',label='with diffusion')
print('mesh Peclet number = ',u*dx/K)

# finalize the plot
plt.axis(np.array([0,lenD,-5,18]))
plt.xlabel('distance [m]')
plt.ylabel('concentration [mg/m3]')
plt.legend(loc = "upper left")
plt.grid()

## Part IV: numerical stability

In this part of the assignment we will explore the stability of the various schemes.

>**Task 4.1**
>
>We study the stability (and accuracy) of the following three schemes:
>- the FTCS scheme,
>- the FTBS scheme, and
>- Crank-Nicolson with central differences.
>
>Do the following:
>
>1. Try to find the maximum time step for each scheme such that the solution is about to be stable.
>2. Plot the stable solution of the given scheme in a graph and write down the time step you found. You will need these results later for the stability analysis.
>
>Use the parameters from the last run in the previous question, except $K = 125$ $m^{2}$/s.
>
>
>1. Which of the schemes is/are unconditionally stable? 
>2. Which of the schemes is the most accurate one? Why?
>3. Which of the numerical solutions has the largest maximum? Why?
>


In [ ]:
# time step based on stability of FTCS
K = 125.
dt2 = np.floor(0.5 * dx*dx / K)

print('maximum time step = ',dt2)
ce = advdiff(u,K,c0,lenD,dt2,Tend,'central','theta',0.)
N2 = int(Tend/dt2)
plt.plot(x,ce[N2,:],'k',label='stable FTCS')

# time step based on stability of FTBS
### YOUR CODE HERE ###

# time step based on stability of trapezoidal rule/central differences
### YOUR CODE HERE ###

> By Marcel Zijlema, Delft University of Technology. CC BY 4.0, more info [on the Credits page of Workbook](https://mude.citg.tudelft.nl/workbook-2026/credits.html).